# KV-Cache, Memory Bandwidth, And Inference Complexity

This notebook isolates the decode-time systems bottleneck that appears after exact attention math is already understood. The main point is simple: prefill and decode are not the same workload, and KV-cache design changes the memory footprint and memory-bandwidth cost of autoregressive generation.

## Motivation

Earlier notebooks focused on the attention map itself. At inference time, a different bottleneck becomes visible: storing and repeatedly reading the cached keys and values for every layer. Multi-query attention (MQA) and grouped-query attention (GQA) reduce that cache size by sharing keys and values across multiple query heads [@fast_transformer_decoding_2019; @gqa_2023]. FlashAttention is relevant here as an IO-aware exact-attention kernel, but it does not change the softmax-attention equations or the KV-cache accounting derived below [@flashattention_2022; @flashattention2_2023].

## Hypothesis

For fixed layer count, batch size, head dimension, and query-head count, KV-cache bytes should scale linearly with context length and with the number of key/value heads. That means MHA should use the most cache, MQA the least, and GQA should sit in between. The prefill stage should show quadratic score-work growth in prompt length, while each decode step should add a constant number of cache bytes but read a linearly growing cache prefix.

## Baseline

The baseline is standard multi-head attention (MHA), where every query head owns its own key and value head. We compare that against MQA, which uses a single shared key/value head, and a small GQA example with fewer key/value heads than query heads.

## Metric

The primary metrics are exact byte counts: cache bytes per token, cache bytes per layer, total cache bytes, and decode-time KV read bytes per generated token. For complexity separation we also track the number of attention score elements touched during prefill versus decode.

## Mathematical derivation

For one decoder layer, each cached token stores one key vector and one value vector for every key/value head. If batch size is `B`, the number of key/value heads is `H_kv`, head dimension is `d_h`, and each stored element uses `s` bytes, then the cache growth per token per layer is

$$b_{\text{token, layer}} = 2 B H_{kv} d_h s.$$

The factor of `2` is for keys and values. After `T` cached tokens, cache bytes per layer are

$$b_{\text{layer}}(T) = T \cdot 2 B H_{kv} d_h s,$$

and across `L` transformer layers the total cache is

$$b_{\text{total}}(T) = L T \cdot 2 B H_{kv} d_h s.$$

The only difference among MHA, MQA, and GQA in this accounting is `H_kv`:

- MHA: `H_kv = H_q`
- MQA: `H_kv = 1`
- GQA: `1 < H_kv < H_q`, with query heads partitioned into groups

For one decode step at current context length `T`, the new query attends to `T` cached positions, so the score workload is approximately `B H_q T`. During prefill with prompt length `T_p`, all prompt queries attend to all prompt keys, so the score workload is approximately `B H_q T_p^2`. That is the systems reason to keep prefill and decode separate even though both use exact attention math.

In [ ]:
from pathlib import Path
import sys

import torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from llm_from_scratch.research.inference import (
    KVCacheConfig,
    kv_cache_bytes,
    kv_cache_bytes_per_layer,
    kv_cache_bytes_per_token,
    resolve_num_kv_heads,
    simulate_autoregressive_inference,
)

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

## PyTorch implementation

The research helper stays arithmetic-first: it resolves how many key/value heads a variant stores, computes exact byte counts, and emits a tiny autoregressive trace with one prefill row plus one row per generated token. We use PyTorch here for dtype sizes and for a small tensor-backed sanity check of the formulas.

In [ ]:
def mib(num_bytes: int) -> float:
    return num_bytes / (1024 ** 2)

dtype_sizes = {
    'fp32': torch.tensor([], dtype=torch.float32).element_size(),
    'fp16': torch.tensor([], dtype=torch.float16).element_size(),
    'int8_like': torch.tensor([], dtype=torch.int8).element_size(),
}

model_shape = {
    'num_layers': 24,
    'num_query_heads': 16,
    'head_dim': 128,
    'batch_size': 1,
    'seq_len': 2048,
}

variant_specs = [
    ('MHA', {'attention_variant': 'mha', 'num_kv_heads': None}),
    ('GQA-4', {'attention_variant': 'gqa', 'num_kv_heads': 4}),
    ('MQA', {'attention_variant': 'mqa', 'num_kv_heads': None}),
]

summary_rows = []
for variant_name, variant_kwargs in variant_specs:
    num_kv_heads = resolve_num_kv_heads(
        num_query_heads=model_shape['num_query_heads'],
        attention_variant=variant_kwargs['attention_variant'],
        num_kv_heads=variant_kwargs['num_kv_heads'],
    )
    for dtype_name, bytes_per_element in dtype_sizes.items():
        per_token_layer = kv_cache_bytes_per_token(
            batch_size=model_shape['batch_size'],
            num_kv_heads=num_kv_heads,
            head_dim=model_shape['head_dim'],
            bytes_per_element=bytes_per_element,
        )
        per_layer = kv_cache_bytes_per_layer(
            seq_len=model_shape['seq_len'],
            batch_size=model_shape['batch_size'],
            num_kv_heads=num_kv_heads,
            head_dim=model_shape['head_dim'],
            bytes_per_element=bytes_per_element,
        )
        total = kv_cache_bytes(
            seq_len=model_shape['seq_len'],
            num_layers=model_shape['num_layers'],
            batch_size=model_shape['batch_size'],
            num_query_heads=model_shape['num_query_heads'],
            head_dim=model_shape['head_dim'],
            bytes_per_element=bytes_per_element,
            attention_variant=variant_kwargs['attention_variant'],
            num_kv_heads=variant_kwargs['num_kv_heads'],
        )
        summary_rows.append({
            'variant': variant_name,
            'dtype': dtype_name,
            'num_kv_heads': num_kv_heads,
            'cache_bytes_per_token_per_layer': per_token_layer,
            'cache_mib_per_layer_at_2048': round(mib(per_layer), 3),
            'cache_mib_total_at_2048': round(mib(total), 3),
        })

summary_rows

The summary table makes the storage rule visible. For the same model width and context length, the only thing changing is the number of stored key/value heads. That is why GQA and MQA lower both total cache capacity and the bytes added by each newly generated token.

## Numerical checks

The next cell checks the formulas against tiny explicit PyTorch tensors. The tensor allocation is intentionally small so that the test is about accounting, not about kernel speed.

In [ ]:
def actual_kv_tensor_bytes(
    *,
    seq_len: int,
    batch_size: int,
    num_kv_heads: int,
    head_dim: int,
    dtype: torch.dtype,
) -> int:
    key_cache = torch.zeros(batch_size, num_kv_heads, seq_len, head_dim, dtype=dtype)
    value_cache = torch.zeros(batch_size, num_kv_heads, seq_len, head_dim, dtype=dtype)
    return key_cache.numel() * key_cache.element_size() + value_cache.numel() * value_cache.element_size()

check_rows = []
for variant_name, variant_kwargs in variant_specs:
    num_kv_heads = resolve_num_kv_heads(
        num_query_heads=8,
        attention_variant=variant_kwargs['attention_variant'],
        num_kv_heads=variant_kwargs['num_kv_heads'] if variant_kwargs['attention_variant'] != 'gqa' else 2,
    )
    for dtype in (torch.float32, torch.float16, torch.int8):
        bytes_per_element = torch.tensor([], dtype=dtype).element_size()
        formula = kv_cache_bytes_per_layer(
            seq_len=5,
            batch_size=2,
            num_kv_heads=num_kv_heads,
            head_dim=4,
            bytes_per_element=bytes_per_element,
        )
        actual = actual_kv_tensor_bytes(
            seq_len=5,
            batch_size=2,
            num_kv_heads=num_kv_heads,
            head_dim=4,
            dtype=dtype,
        )
        assert actual == formula
        check_rows.append({
            'variant': variant_name,
            'dtype': str(dtype).replace('torch.', ''),
            'num_kv_heads': num_kv_heads,
            'bytes': actual,
        })

check_rows

The next check uses the decode simulator. It keeps prefill as a single quadratic-work event, then emits one row per generated token where cache growth is constant but cache-read bandwidth grows with context length.

In [ ]:
decode_config = KVCacheConfig(
    num_layers=24,
    num_query_heads=16,
    num_kv_heads=4,
    head_dim=128,
    batch_size=1,
    bytes_per_element=dtype_sizes['fp16'],
    attention_variant='gqa',
)

trace = simulate_autoregressive_inference(
    config=decode_config,
    prompt_len=1024,
    generated_tokens=4,
)

assert trace[0].phase == 'prefill'
assert all(step.phase == 'decode' for step in trace[1:])
assert trace[1].cache_growth_bytes_total == trace[2].cache_growth_bytes_total
assert trace[2].kv_read_bytes_total > trace[1].kv_read_bytes_total

prefill_row = {
    'phase': trace[0].phase,
    'prompt_tokens': trace[0].cache_tokens,
    'attention_score_elements': trace[0].attention_score_elements,
    'cache_mib_per_layer_after_prefill': round(mib(trace[0].cache_bytes_per_layer), 3),
    'cache_mib_total_after_prefill': round(mib(trace[0].cache_bytes_total), 3),
}

decode_rows = [
    {
        'generated_token': step.step_index,
        'context_before_step': step.context_tokens_before_step,
        'cache_tokens_after_step': step.cache_tokens,
        'cache_kib_per_layer_after_step': round(step.cache_bytes_per_layer / 1024, 2),
        'cache_kib_growth_total': round(step.cache_growth_bytes_total / 1024, 2),
        'kv_read_kib_total': round(step.kv_read_bytes_total / 1024, 2),
        'attention_score_elements': step.attention_score_elements,
    }
    for step in trace[1:]
]

prefill_row, decode_rows

This is the acceptance-level view for the notebook: cache memory is reported by layer and by generated token. The constant `cache_kib_growth_total` column is the per-token write increment, while `kv_read_kib_total` grows because each new token must read a longer cache prefix.

## Ablations

We can separate storage growth from score-work growth with two tiny sweeps: one over attention variants at fixed context length, and one over prompt length for the first decode step.

In [ ]:
variant_ablation = []
for variant_name, variant_kwargs in variant_specs:
    config = KVCacheConfig(
        num_layers=24,
        num_query_heads=16,
        num_kv_heads=variant_kwargs['num_kv_heads'],
        head_dim=128,
        batch_size=1,
        bytes_per_element=dtype_sizes['fp16'],
        attention_variant=variant_kwargs['attention_variant'],
    )
    step = simulate_autoregressive_inference(config=config, prompt_len=2048, generated_tokens=1)[1]
    variant_ablation.append({
        'variant': variant_name,
        'num_kv_heads': resolve_num_kv_heads(
            num_query_heads=config.num_query_heads,
            attention_variant=config.attention_variant,
            num_kv_heads=config.num_kv_heads,
        ),
        'decode_kv_read_mib_total': round(mib(step.kv_read_bytes_total), 3),
        'decode_cache_growth_kib_total': round(step.cache_growth_bytes_total / 1024, 2),
    })

context_ablation = []
for prompt_len in (128, 512, 2048, 4096):
    step0, step1 = simulate_autoregressive_inference(
        config=decode_config,
        prompt_len=prompt_len,
        generated_tokens=1,
    )
    context_ablation.append({
        'prompt_len': prompt_len,
        'prefill_score_elements': step0.attention_score_elements,
        'decode_score_elements_first_token': step1.attention_score_elements,
        'decode_kv_read_mib_total_first_token': round(mib(step1.kv_read_bytes_total), 3),
    })

variant_ablation, context_ablation

## Assumptions

The accounting assumes dense storage for cached keys and values, no extra metadata, and one stored element size shared across K and V. It also treats decode-time KV read bytes as proportional to the stored cache size, which is the right first-order systems model for this toy notebook.

## Risks

Real inference kernels can fuse loads, tile reads, or quantize KV-cache values with extra scale tensors, so production bandwidth may differ from this arithmetic model. FlashAttention also reduces IO pressure for exact attention execution, but that should not be read as a change to the underlying attention math or as a claim that KV-cache costs disappear [@flashattention_2022; @flashattention2_2023].

## Failure criteria

Treat this notebook as failed if the tensor-backed byte check disagrees with the formulas, if MHA/GQA/MQA do not order by cache size as expected, or if the simulator fails to show the intended split: quadratic prefill score work versus linearly growing decode-time cache reads.

## Exercises

1. Re-derive the per-token and per-layer KV-cache formulas and identify where each factor comes from.
2. Why does GQA reduce cache pressure without changing the number of query heads?
3. Which decode-time quantity grows with context length in this notebook even when the per-token cache write stays constant?

Companion exercise files:

- `exercises/research/18_kv_cache_inference.md`
- `exercises/research/solutions/18_kv_cache_inference_solutions.md`

## References

- Shazeer, *Fast Transformer Decoding: One Write-Head is All You Need* [@fast_transformer_decoding_2019].
- Ainslie et al., *GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints* [@gqa_2023].
- Dao et al., *FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness* [@flashattention_2022].
- Dao, *FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning* [@flashattention2_2023].